# Network analytics — which entities and relationships are impactful?

Once you have a knowledge graph, the next question is usually *which nodes matter the most*. This notebook pulls the graph that Tutorial 1 built into NetworkX and runs four classic network metrics:

| Metric | Question it answers |
|---|---|
| Degree centrality | Who has the most direct connections? |
| PageRank | Who is *important* via a recursive popularity walk? |
| Betweenness centrality | Who sits on the shortest paths between other nodes? |
| Community detection | Which nodes naturally cluster together? |

Why NetworkX and not Neo4j GDS? GDS is great for production scale-out, but it requires graph projections, plugin installs, and Cypher familiarity. NetworkX is one Python import, runs anywhere, and the algorithms are exactly the same. When the graph outgrows a single Python process, swap the algorithm calls for `gds.pageRank.stream(...)` etc. — the *concepts* you learn here transfer 1:1.

**Prerequisite:** Run Tutorial 1 first to populate the `finder_tutorial` workspace in Neo4j. This notebook reads from that workspace; it does not re-extract.

## Setup

In [ ]:
import json
import os
import sys
from collections import Counter, defaultdict
from pathlib import Path

from dotenv import load_dotenv

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "seocho").is_dir() and (p / "examples").is_dir():
            return p
        p = p.parent
    return Path.cwd()

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://tutorials-neo4j:7687")
NEO4J_USER     = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "tutorialspw")
WORKSPACE_ID   = os.environ.get("SEOCHO_NETWORK_WORKSPACE", "finder_tutorial")

print(f"Neo4j @ {NEO4J_URI}")
print(f"Reading workspace: {WORKSPACE_ID}")

In [ ]:
from seocho.store.graph import Neo4jGraphStore
from examples.finder.lib.graph_viz import fetch_lpg_subgraph

graph_store = Neo4jGraphStore(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
nodes, relationships = fetch_lpg_subgraph(
    graph_store, workspace_id=WORKSPACE_ID, limit=500,
)
if not nodes:
    raise RuntimeError(
        f"Workspace '{WORKSPACE_ID}' is empty. Run Tutorial 1 first "
        "(01_vector_vs_graph_rag.ipynb) so it populates this workspace."
    )
print(f"Pulled {len(nodes)} nodes, {len(relationships)} relationships")

In [ ]:
import networkx as nx

g = nx.DiGraph()
for n in nodes:
    nid = n["id"]
    g.add_node(
        nid,
        label=n.get("label", "Entity"),
        name=str(n.get("properties", {}).get("name") or nid),
    )
for r in relationships:
    if r.get("source") and r.get("target"):
        g.add_edge(r["source"], r["target"], type=r.get("type", "REL"))

print(f"NetworkX DiGraph: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")
print(f"Density:           {nx.density(g):.4f}")
print(f"Components (weak): {nx.number_weakly_connected_components(g)}")

## 1. Degree centrality — who has the most direct connections?

The simplest *important node* signal: how many edges touch it. A `Company` that's mentioned by every FinDER document will rank high. Useful for spotting hub entities in your graph.

In [ ]:
ud = g.to_undirected()  # degree counts both incoming and outgoing
deg = dict(ud.degree())

def top_n(scores: dict, n: int = 10):
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:n]

print(f"{'rank':4s}  {'name':40s}  {'label':18s}  {'degree':>6s}")
print("-" * 80)
for i, (nid, d) in enumerate(top_n(deg), 1):
    data = g.nodes[nid]
    print(f"{i:>4}  {data['name'][:40]:40s}  {data['label']:18s}  {d:>6d}")

## 2. PageRank — recursive importance

Degree treats every neighbor equally. PageRank doesn't: a node connected to highly-ranked nodes gets a bigger boost than a node connected to obscure ones. The classic intuition: a small company mentioned alongside Apple, Microsoft, and Berkshire Hathaway looks more important than one mentioned alongside three obscure ones.

In [ ]:
pr = nx.pagerank(g, alpha=0.85)
print(f"{'rank':4s}  {'name':40s}  {'label':18s}  {'pagerank':>9s}")
print("-" * 80)
for i, (nid, score) in enumerate(top_n(pr), 1):
    data = g.nodes[nid]
    print(f"{i:>4}  {data['name'][:40]:40s}  {data['label']:18s}  {score:>9.4f}")

## 3. Betweenness centrality — bridges between communities

Betweenness asks a different question: how often does this node lie on the shortest path between *other* pairs of nodes? High-betweenness nodes are the bridges — remove them and the graph fragments. In FinDER, an executive who connects two otherwise-disjoint companies will rank here.

In [ ]:
btw = nx.betweenness_centrality(g)
print(f"{'rank':4s}  {'name':40s}  {'label':18s}  {'betweenness':>11s}")
print("-" * 80)
for i, (nid, score) in enumerate(top_n(btw), 1):
    data = g.nodes[nid]
    print(f"{i:>4}  {data['name'][:40]:40s}  {data['label']:18s}  {score:>11.4f}")

## 4. Community detection — natural clusters

Connected groups in the graph often correspond to meaningful clusters — a company plus its executives, financial metrics, and lawsuits forms one community. We use **label propagation** here because it's parameter-free and ships with NetworkX. For larger graphs swap in Louvain (`pip install python-louvain`) or Leiden.

In [ ]:
from networkx.algorithms.community import label_propagation_communities

communities = list(label_propagation_communities(ud))
communities.sort(key=len, reverse=True)

print(f"Detected {len(communities)} communities. Top 5 by size:")
for i, c in enumerate(communities[:5], 1):
    sample_names = [g.nodes[n]["name"][:25] for n in list(c)[:6]]
    print(f"  community {i:>2}  size={len(c):>3}  members: {', '.join(sample_names)}"
          + (" …" if len(c) > 6 else ""))

# Map node -> community index for the visualization next
node_to_community = {n: i for i, c in enumerate(communities) for n in c}

## 5. Visualize — color by community, size by PageRank

Two signals at once: communities tell you *which slice of the world* a node belongs to (color); PageRank tells you *how prominent* it is in that slice (size).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(13, 9))
pos = nx.spring_layout(g, seed=42, k=0.6)

palette = plt.cm.tab20.colors
node_colors = [palette[node_to_community.get(n, 0) % len(palette)] for n in g.nodes()]
node_sizes  = [200 + 8000 * pr.get(n, 0.0) for n in g.nodes()]

nx.draw_networkx_nodes(g, pos, node_color=node_colors, node_size=node_sizes, alpha=0.92, ax=ax)
nx.draw_networkx_edges(g, pos, alpha=0.3, arrows=True, ax=ax)

# Label only the top-N PageRank nodes — full labels would be unreadable
TOP_LABELS = 12
labelled = {nid: g.nodes[nid]["name"][:22] for nid, _ in top_n(pr, TOP_LABELS)}
nx.draw_networkx_labels(g, pos, labels=labelled, font_size=8, font_weight="bold", ax=ax)

ax.set_title(
    f"FinDER graph — color: community ({len(communities)} clusters)  ·  size: PageRank",
    fontsize=11,
)
ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Edge importance — which relationship types carry the most signal?

Edges have *types* (`EMPLOYS`, `REPORTED`, `MENTIONS`, …). Which type appears most often is a quick proxy for which relationships drive the structure of the graph. Multi-edges between the same pair of nodes also indicate semantic redundancy — worth pruning during ontology design.

In [ ]:
rel_type_counts = Counter(d.get("type", "REL") for _, _, d in g.edges(data=True))

print(f"{'rank':4s}  {'rel type':24s}  {'count':>6s}  {'share':>6s}")
print("-" * 60)
total = sum(rel_type_counts.values())
for i, (rtype, cnt) in enumerate(rel_type_counts.most_common(), 1):
    print(f"{i:>4}  {rtype:24s}  {cnt:>6d}  {cnt/total:>5.0%}")

# Flag any pair of nodes connected by 2+ different relationship types —
# usually a sign of overlapping semantics in the ontology.
pair_types = defaultdict(set)
for u, v, d in g.edges(data=True):
    pair_types[(u, v)].add(d.get("type", "REL"))
redundant = [(uv, types) for uv, types in pair_types.items() if len(types) >= 2]
print(f"\nNode pairs with multiple distinct edge types: {len(redundant)}")
for (u, v), types in redundant[:5]:
    print(f"  {g.nodes[u]['name'][:25]} -> {g.nodes[v]['name'][:25]}  via {sorted(types)}")

## What to take away

- **Degree** is fast and intuitive. Use it as a sanity check — your most-mentioned company should be near the top.
- **PageRank** is degree's smarter cousin. Important when the graph has hubs of hubs (markets where a few players dominate).
- **Betweenness** identifies bridges. Removing high-betweenness nodes disconnects the graph; they're often the documents/executives that link otherwise-separate sub-corpora.
- **Community detection** is your shortcut to *summarization*. A single sentence per community often captures most of what the graph is about.
- **Edge-type frequency** is a cheap sanity check on the ontology. If 95% of your edges are `MENTIONS`, your ontology is too coarse — most facts collapsed into one bucket. If 50 different rel types each have <2% share, you over-fragmented.

**Production scale-up.** When the graph outgrows a single Python process, the same algorithms are available natively in Neo4j Graph Data Science:

```cypher
CALL gds.pageRank.stream({nodeProjection: '*', relationshipProjection: '*'})
YIELD nodeId, score RETURN gds.util.asNode(nodeId).name AS name, score ORDER BY score DESC
```

GDS Community Edition has node-count limits; for graphs above a few million nodes you'll want GDS Enterprise or a dedicated graph-analytics tool. The intuitions you built here transfer.